# Q4 — New dataset: **ImageNet‑R (few‑shot)** — *Train & Checkpoint*

This notebook mirrors your EuroSAT flow but runs **MambaVision** (frozen + linear probe) and **three baselines** with linear probes:
- ResNet‑18
- ViT‑B/16
- ConvNeXt‑Tiny

It also adds a storage switch (**Colab Drive** vs **local**) for checkpoints and artifacts.


In [1]:
!pip install mamba_ssm timm transformers Pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.8/113.8 kB 5.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (180 kB)
  Created wheel for mamba_ssm: filename=mamba_ssm-2.2.5-cp312-cp312-linux_x86_64.whl size=532566033 sha256=c8b65fcabfb49a94456c9971619007218e4073f19a84fb6b3894f33d43bee4a1
  Stored in directory: /root/.cache/pip/wheels/21/55/c4/85b634055d6a9b599d27f5cbeacf353c6c532d8e2d8769960b
Successfully built mamba_ssm


In [2]:
# --- Environment setup  ---


import os, sys, json, time, random, math, statistics
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
from torchvision import transforms, datasets
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedShuffleSplit
from PIL import Image
import matplotlib.pyplot as plt

from transformers import AutoModel   # MambaVision frozen backbone
from timm.data.transforms_factory import create_transform

print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("Using CUDA?", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"


PyTorch: 2.8.0+cu126
torchvision: 0.23.0+cu126
Using CUDA? True


## Storage switch (Colab Drive vs local)

Set `STORAGE = "colab"` to save checkpoints and results to your Google Drive when running in Colab.  
Set `STORAGE = "local"` to save alongside the notebook.


In [4]:
# --- Storage configuration ---
STORAGE = "colab"   # <-- change to "colab" when running in Google Colab

def get_artifacts_dir():
    in_colab = False
    try:
        import google.colab  # type: ignore
        in_colab = True
    except Exception:
        pass
    if STORAGE.lower() == "colab" and in_colab:
        from google.colab import drive  # type: ignore
        drive.mount('/content/drive', force_remount=True)
        base = Path('/content/drive/MyDrive/MambaVision_Q4_Artifacts4')
    else:
        base = Path('./artifacts2')
    (base / 'checkpoints').mkdir(parents=True, exist_ok=True)
    (base / 'results').mkdir(parents=True, exist_ok=True)
    (base / 'splits').mkdir(parents=True, exist_ok=True)
    return base

ARTIFACTS_DIR = get_artifacts_dir()
print("Artifacts root:", ARTIFACTS_DIR)


Mounted at /content/drive
Artifacts root: /content/drive/MyDrive/MambaVision_Q4_Artifacts4


## Dataset: **ImageNet‑R (Renditions)** — build a *few‑shot* split

- Point `IMAGENET_R_ROOT` to your extracted ImageNet‑R directory (standard `ImageFolder` layout).
- We create a **K‑shot** split per class (default: 5‑shot for train and 5‑shot for val). The remaining images form the test set.
- MambaVision and TorchVision models use **their own** transforms (MambaVision via `config.mean/std/crop_*`).

> Tip: Download and extract the dataset once; you can reuse the path locally or in Drive.


In [5]:
import tarfile, urllib.request, os

url = "https://people.eecs.berkeley.edu/~hendrycks/imagenet-r.tar"
os.makedirs("data", exist_ok=True)
tar_path = "data/imagenet-r.tar"

urllib.request.urlretrieve(url, tar_path)
with tarfile.open(tar_path, "r:") as t:
    t.extractall("data")   # -> data/imagenet-r/
print("Done. Extracted to data/imagenet-r")


/tmp/ipython-input-1400197689.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall("data")   # -> data/imagenet-r/


Done. Extracted to data/imagenet-r


In [6]:
# --- Few-shot split on ImageNet-R ---
IMAGENET_R_ROOT = "/content/data/imagenet-r"   # TODO: change this to your local path
K_SHOT_TRAIN = 40
K_SHOT_VAL = 10
IMG_SIZE = 224
BATCH_SIZE = 128
NUM_WORKERS = 2
SEED = 42

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# Load once (no transforms) to access labels & indices
base = datasets.ImageFolder(IMAGENET_R_ROOT, transform=None)
class_to_idx = base.class_to_idx
idx_to_class = {v:k for k,v in class_to_idx.items()}
NUM_CLASSES = len(class_to_idx)
print("Classes:", NUM_CLASSES)

# Build per-class index lists
per_class_indices = {c: [] for c in range(NUM_CLASSES)}
for i, (_, y) in enumerate(base.samples):
    per_class_indices[y].append(i)
for c in per_class_indices:
    rng = np.random.default_rng(SEED)
    rng.shuffle(per_class_indices[c])

def make_split(k_train=5, k_val=5):
    train_idx, val_idx, test_idx = [], [], []
    for c, idxs in per_class_indices.items():
        n = len(idxs)
        k1 = min(k_train, n)
        k2 = min(k_val, max(0, n - k1))
        tr = idxs[:k1]
        va = idxs[k1:k1+k2]
        te = idxs[k1+k2:]
        train_idx += tr; val_idx += va; test_idx += te
    return train_idx, val_idx, test_idx

train_idx, val_idx, test_idx = make_split(K_SHOT_TRAIN, K_SHOT_VAL)
print("Split sizes:", len(train_idx), len(val_idx), len(test_idx))

# Save split indices for Notebook 2 reuse
with open(ARTIFACTS_DIR / "splits" / "indices.json", "w") as f:
    json.dump({"train": train_idx, "val": val_idx, "test": test_idx}, f, indent=2)


Classes: 200
Split sizes: 8000 2000 20000


## Load **MambaVision** backbone + transforms

We use the **feature‑extraction** variant (`AutoModel`) and freeze it.  
The forward returns `(out_avg_pool, features)`. We train a small linear head on top.


In [7]:
# --- Load MambaVision backbone & its transforms ---
mamba_backbone = AutoModel.from_pretrained("nvidia/MambaVision-T-1K", trust_remote_code=True).to(device).eval()

input_resolution = (3, IMG_SIZE, IMG_SIZE)
mamba_train_tf = create_transform(
    input_size=input_resolution,
    is_training=True,
    mean=mamba_backbone.config.mean,
    std=mamba_backbone.config.std,
    crop_mode=mamba_backbone.config.crop_mode,
    crop_pct=mamba_backbone.config.crop_pct,
)
mamba_eval_tf = create_transform(
    input_size=input_resolution,
    is_training=False,
    mean=mamba_backbone.config.mean,
    std=mamba_backbone.config.std,
    crop_mode=mamba_backbone.config.crop_mode,
    crop_pct=mamba_backbone.config.crop_pct,
)

# TorchVision models share ImageNet normalization
tv_train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])
tv_eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

# Build view datasets with transforms and chosen indices
def make_subset(transform, indices):
    ds = datasets.ImageFolder(IMAGENET_R_ROOT, transform=transform)
    return Subset(ds, indices)

train_mamba = make_subset(mamba_train_tf, train_idx)
val_mamba   = make_subset(mamba_eval_tf,  val_idx)
test_mamba  = make_subset(mamba_eval_tf,  test_idx)

train_tv = make_subset(tv_train_tf, train_idx)
val_tv   = make_subset(tv_eval_tf,  val_idx)
test_tv  = make_subset(tv_eval_tf,  test_idx)

train_loader_m = DataLoader(train_mamba, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader_m   = DataLoader(val_mamba,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader_m  = DataLoader(test_mamba,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

train_loader_tv = DataLoader(train_tv, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader_tv   = DataLoader(val_tv,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader_tv  = DataLoader(test_tv,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

len(train_mamba), len(val_mamba), len(test_mamba)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

configuration_mambavision.py:   0%|          | 0.00/625 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/nvidia/MambaVision-T-1K:
- configuration_mambavision.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_mambavision.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/nvidia/MambaVision-T-1K:
- modeling_mambavision.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


model.safetensors:   0%|          | 0.00/127M [00:00<?, ?B/s]

(8000, 2000, 20000)

## Linear‑probe heads (frozen backbones)


In [8]:
# --- Linear probe heads ---
class MambaLinearProbe(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        for p in self.backbone.parameters():
            p.requires_grad = False
        # Infer feature dimension
        with torch.no_grad():
            dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE).to(next(self.backbone.parameters()).device)
            out_avg_pool, _ = self.backbone(dummy)
            feat_dim = out_avg_pool.shape[1]
        self.head = nn.Linear(feat_dim, num_classes)

    def forward(self, x):
        with torch.no_grad():
            out_avg_pool, _ = self.backbone(x)
        return self.head(out_avg_pool)

# def make_resnet18_linear(num_classes):
#     weights = torchvision.models.ResNet18_Weights.DEFAULT
#     net = torchvision.models.resnet18(weights=weights)
#     for p in net.parameters():
#         p.requires_grad = False
#     in_features = net.fc.in_features
#     net.fc = nn.Linear(in_features, num_classes)
#     return net

def make_vit_b16_linear(num_classes):
    # TorchVision ViT-B/16
    weights = torchvision.models.ViT_B_16_Weights.DEFAULT
    net = torchvision.models.vit_b_16(weights=weights)
    for p in net.parameters():
        p.requires_grad = False
    in_features = net.heads.head.in_features
    net.heads.head = nn.Linear(in_features, num_classes)
    return net

def make_convnext_tiny_linear(num_classes):
    weights = torchvision.models.ConvNeXt_Tiny_Weights.DEFAULT
    net = torchvision.models.convnext_tiny(weights=weights)
    for p in net.parameters():
        p.requires_grad = False
    in_features = net.classifier[2].in_features
    net.classifier[2] = nn.Linear(in_features, num_classes)
    return net

mamba_probe   = MambaLinearProbe(mamba_backbone, NUM_CLASSES).to(device)
# resnet_probe  = make_resnet18_linear(NUM_CLASSES).to(device)
vit_probe     = make_vit_b16_linear(NUM_CLASSES).to(device)
convnext_probe= make_convnext_tiny_linear(NUM_CLASSES).to(device)


Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:02<00:00, 118MB/s]


Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 169MB/s]


## Training helpers


In [9]:
def accuracy(output, target):
    with torch.no_grad():
        pred = output.argmax(dim=1)
        return (pred == target).float().mean().item() * 100.0

def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total, correct, loss_sum = 0, 0, 0.0
    crit = nn.CrossEntropyLoss()
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        if is_train:
            optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = crit(logits, y)
        if is_train:
            loss.backward()
            optimizer.step()
        loss_sum += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return loss_sum / max(1,total), 100.0 * correct / max(1,total)

def train_linear_probe(tag, model, train_loader, val_loader, epochs=10, lr=1e-2, wd=1e-4):
    ckpt_dir = ARTIFACTS_DIR / "checkpoints" / tag
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=lr, momentum=0.9, weight_decay=wd, nesterov=True)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_val, best_path = -1.0, None
    for ep in range(1, epochs+1):
        tr_loss, tr_acc = run_epoch(model, train_loader, optimizer)
        va_loss, va_acc = run_epoch(model, val_loader, optimizer=None)
        scheduler.step()
        print(f"[{tag}] epoch {ep:02d}/{epochs}  train {tr_acc:5.2f}%  val {va_acc:5.2f}%")
        if va_acc > best_val:
            best_val = va_acc
            best_path = ckpt_dir / "best.pt"
            torch.save({"state_dict": model.state_dict(), "val_acc": va_acc}, best_path)
    return best_val, best_path


## Train all 4 linear probes and evaluate on test


In [10]:
EPOCHS = 5
LR = 1e-2
WD = 1e-4

results = []

jobs = [
    ("mambavision_lp", mamba_probe, train_loader_m, val_loader_m, test_loader_m),
    # ("resnet18_lp",    resnet_probe, train_loader_tv, val_loader_tv, test_loader_tv),
    ("vit_b16_lp",     vit_probe, train_loader_tv, val_loader_tv, test_loader_tv),
    ("convnext_tiny_lp", convnext_probe, train_loader_tv, val_loader_tv, test_loader_tv),
]

for tag, model, tl, vl, tel in jobs:
    model = model.to(device)
    best_val, ckpt_path = train_linear_probe(tag, model, tl, vl, epochs=EPOCHS, lr=LR, wd=WD)
    # Load best & test
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state["state_dict"])
    te_loss, te_acc = run_epoch(model, tel, optimizer=None)
    print(f"[{tag}] Best Val {best_val:.2f} | Test {te_acc:.2f}")
    results.append({"model": tag, "best_val_acc": best_val, "test_acc": te_acc, "ckpt": str(ckpt_path)})

# Save results & config like your EuroSAT notebook
df = pd.DataFrame(results)
(ARTIFACTS_DIR / "results").mkdir(exist_ok=True, parents=True)
df.to_csv(ARTIFACTS_DIR / "results" / "q4_results_imagenet_r.csv", index=False)

with open(ARTIFACTS_DIR / "results" / "q4_config.json", "w") as f:
    json.dump({
        "seed": SEED,
        "k_shot_train": K_SHOT_TRAIN,
        "k_shot_val": K_SHOT_VAL,
        "img_size": IMG_SIZE,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "optimizer": "SGD(momentum=0.9)",
        "lr": LR,
        "weight_decay": WD,
        "models": [r["model"] for r in results],
        "artifacts_dir": str(ARTIFACTS_DIR)
    }, f, indent=2)

print("Saved metrics to:", ARTIFACTS_DIR / "results" / "q4_results_imagenet_r.csv")
df


[mambavision_lp] epoch 01/5  train  2.52%  val 12.70%
[mambavision_lp] epoch 02/5  train 17.39%  val 32.95%
[mambavision_lp] epoch 03/5  train 28.85%  val 39.90%
[mambavision_lp] epoch 04/5  train 32.45%  val 42.15%
[mambavision_lp] epoch 05/5  train 34.38%  val 42.80%
[mambavision_lp] Best Val 42.80 | Test 38.26
[vit_b16_lp] epoch 01/5  train 14.95%  val 36.60%
[vit_b16_lp] epoch 02/5  train 47.89%  val 45.90%
[vit_b16_lp] epoch 03/5  train 56.61%  val 48.75%
[vit_b16_lp] epoch 04/5  train 59.96%  val 49.70%
[vit_b16_lp] epoch 05/5  train 61.67%  val 49.85%
[vit_b16_lp] Best Val 49.85 | Test 47.10
[convnext_tiny_lp] epoch 01/5  train  7.84%  val 26.15%
[convnext_tiny_lp] epoch 02/5  train 39.38%  val 41.35%
[convnext_tiny_lp] epoch 03/5  train 51.33%  val 45.50%
[convnext_tiny_lp] epoch 04/5  train 55.89%  val 46.90%
[convnext_tiny_lp] epoch 05/5  train 57.54%  val 47.35%
[convnext_tiny_lp] Best Val 47.35 | Test 43.67
Saved metrics to: /content/drive/MyDrive/MambaVision_Q4_Artifacts4/

,model,best_val_acc,test_acc,ckpt
0,mambavision_lp,42.80,38.26,/content/drive/MyDrive/MambaVision_Q4_Artifact...
1,vit_b16_lp,49.85,47.10,/content/drive/MyDrive/MambaVision_Q4_Artifact...
2,convnext_tiny_lp,47.35,43.67,/content/drive/MyDrive/MambaVision_Q4_Artifact...
